In [1]:
import tensorflow as tf
#import tensorflow_decision_forests as tfdf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder


In [5]:
def traitement_dataset(dataset_df):

### Pour savoir si la personne est dans un groupe ou non
    dataset_df[['group','num_in_the_group']] = dataset_df['PassengerId'].str.split('_', expand=True)
    
### Traitement de la colonne "Cabin" pour en extraire les informations utiles
    dataset_df[['Deck', 'Num', 'Side']] = dataset_df['Cabin'].str.split('/', expand=True)
    side_mapping={"P":1,"S":-1}
    dataset_df['Side'] = dataset_df['Side'].map(side_mapping)

### Encodage des variables catégorielles
    encoder = OneHotEncoder()

## Deck
    deck_encoded = encoder.fit_transform(dataset_df[['Deck']])
    # Convertir le tableau numpy en DataFrame
    deck_df = pd.DataFrame(deck_encoded.toarray(), columns=encoder.get_feature_names_out(['Deck']))
    # Concaténer le DataFrame original avec le nouveau DataFrame
    dataset_df = pd.concat([dataset_df, deck_df], axis=1)

## Destination
    destination_encoded = encoder.fit_transform(dataset_df[['Destination']])
    destination_df = pd.DataFrame(destination_encoded.toarray(), columns=encoder.get_feature_names_out(['Destination']))
    dataset_df = pd.concat([dataset_df, destination_df], axis=1)

## Planet
    planet_encoded = encoder.fit_transform(dataset_df[['HomePlanet']])
    planet_df = pd.DataFrame(planet_encoded.toarray(), columns=encoder.get_feature_names_out(['HomePlanet']))
    dataset_df = pd.concat([dataset_df, planet_df], axis=1)

### Cette partie est je pense inutile car python traite déjà les valeurs booléennes de cette façon
## Mapping by Mael
    cryo_sleep_mapping = {False: 0, True: 1}
    vip_mapping = {False: 0, True: 1}
    transported_mapping = {False: 0, True: 1}

## Modify data according to mappings by Mael
    dataset_df["CryoSleep"] = dataset_df["CryoSleep"].map(cryo_sleep_mapping)
    dataset_df["VIP"] = dataset_df["VIP"].map(vip_mapping)
    dataset_df["Transported"] = dataset_df["Transported"].map(transported_mapping)

## Supression des colonnes
    dataset_df.drop(columns=['PassengerId', 'Name', 'Cabin', 'Deck','Destination','Deck_nan',
                             'Destination_nan','HomePlanet','HomePlanet_nan'],
                     inplace=True)

    return dataset_df

In [6]:
dataset_df = pd.read_csv('train.csv')
dataset_df=traitement_dataset(dataset_df)
dataset_df.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,group,...,Deck_E,Deck_F,Deck_G,Deck_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars
0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0001,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,24.0,0.0,109.0,9.0,25.0,549.0,44.0,1,0002,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,0.0,58.0,1.0,43.0,3576.0,0.0,6715.0,49.0,0,0003,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,0.0,33.0,0.0,0.0,1283.0,371.0,3329.0,193.0,0,0003,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,0.0,16.0,0.0,303.0,70.0,151.0,565.0,2.0,1,0004,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


In [8]:
def is_in_a_group(row):
    if pd.isnull(row['group']):
        return 0
    else:
        return 1

dataset_df['InGroup'] = dataset_df.apply(is_in_a_group, axis=1)

In [9]:
def fill_destination(df):
    # Remplir les valeurs manquantes dans chaque groupe avec la destination la plus fréquente dans ce groupe
    df['Destination'] = df.groupby('Group')['Destination'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))

    # Remplir les valeurs manquantes restantes avec la destination la plus fréquente dans l'ensemble du DataFrame
    df['Destination'] = df['Destination'].fillna(df['Destination'].mode()[0])